### Total revenue per category

In [0]:
%sql
SELECT
    p.category,
    ROUND(
        SUM(
            oi.quantity * oi.unit_price *
            (1 - oi.discount_percent / 100.0)
        ),
        2
    ) AS total_revenue
FROM order_items oi
JOIN products p
ON oi.product_id = p.product_id
GROUP BY p.category
ORDER BY total_revenue DESC;

category,total_revenue
Books,1534578.4
Home,1506599.1
Electronics,1305698.9
Clothing,1057170.34


### Top 10 customers by total order value

In [0]:
%sql
SELECT
    c.customer_id,
    c.customer_name,
    ROUND(
        SUM(
            oi.quantity * oi.unit_price *
            (1 - oi.discount_percent / 100.0)
        ),
        2
    ) AS total_order_value
FROM customers c
JOIN orders o
ON c.customer_id = o.customer_id
JOIN order_items oi
ON o.order_id = oi.order_id
GROUP BY
    c.customer_id,
    c.customer_name
ORDER BY total_order_value DESC
LIMIT 10;

customer_id,customer_name,total_order_value
197,Miss Michelle Pierce,81649.4
87,Brad Allen,73699.46
62,Tiffany Barnes,62630.91
279,Nicole Marquez,59147.04
289,Tony Le,58160.83
23,Rhonda Lee,57760.42
336,Matthew Wright,56546.59
53,Miss Patricia Gibson,56224.54
158,Eric Hill,52265.02
278,Benjamin Frost,51009.54


### Find customers who placed orders but never had any item delivered

In [0]:
%sql
SELECT DISTINCT
    c.customer_id,
    c.customer_name
FROM customers c
JOIN orders o
ON c.customer_id = o.customer_id
WHERE c.customer_id NOT IN
(
    SELECT customer_id
    FROM orders
    WHERE status='DELIVERED'
);

customer_id,customer_name
22,Michael Lewis
31,Lauren Williams
48,Mary Martinez
51,Michael Dixon
62,Tiffany Barnes
128,Tiffany Holloway
139,Kara Davis
159,Ronald Jones
176,Raven Taylor
266,Dale Edwards


### Products that were ordered but had more returns than purchases

In [0]:
%sql
SELECT
    p.product_id,
    p.product_name,
    SUM(CASE WHEN oi.quantity > 0 THEN 1 ELSE 0 END) AS purchases,
    SUM(CASE WHEN oi.quantity < 0 THEN 1 ELSE 0 END) AS returns
FROM products p
JOIN order_items oi
ON p.product_id=oi.product_id
GROUP BY
    p.product_id,
    p.product_name
HAVING returns > purchases;

product_id,product_name,purchases,returns


### Return rate (returned items / total items) per category

In [0]:
%sql
SELECT
    p.category,
    ROUND(
        SUM(CASE WHEN oi.quantity<0 THEN 1 ELSE 0 END)
        *100.0/
        COUNT(*),
        2
    ) AS return_rate
FROM products p
JOIN order_items oi
ON p.product_id=oi.product_id
GROUP BY p.category;

category,return_rate
Home,0.00
Books,0.00
Electronics,0.00
Clothing,0.00


### Running Totals with Window Functions
### 
### Show:
### 
### region_code
### order_date
### daily_revenue
### running_total

In [0]:
%sql
SELECT
    region_code,
    order_date,
    daily_revenue,
    SUM(daily_revenue) OVER(
        PARTITION BY region_code
        ORDER BY order_date
    ) AS running_total
FROM
(
    SELECT
        o.region_code,
        DATE(o.order_date) AS order_date,
        SUM(
            oi.quantity * oi.unit_price *
            (1 - oi.discount_percent/100.0)
        ) AS daily_revenue
    FROM orders o
    JOIN order_items oi
    ON o.order_id=oi.order_id
    GROUP BY
        o.region_code,
        DATE(o.order_date)
);

region_code,order_date,daily_revenue,running_total
EAST,2024-07-27,26281.6977,26281.6977
EAST,2024-08-08,1586.62,27868.3177
EAST,2024-08-16,22225.2512,50093.5689
EAST,2024-08-18,3895.5004,53989.069299999996
EAST,2024-08-27,28738.7328,82727.8021
EAST,2024-08-31,4106.9098,86834.7119
EAST,2024-09-02,7432.246000000001,94266.9579
EAST,2024-09-06,6568.522199999999,100835.48009999999
EAST,2024-09-09,6922.6182,107758.09829999998
EAST,2024-09-17,3092.4216,110850.51989999998


### Ranking with DENSE_RANK
### 
### Show:
### 
### category
### product_name
### total_revenue
### rank_in_category

In [0]:
%sql
SELECT
    category,
    product_name,
    total_revenue,
    DENSE_RANK() OVER(
        PARTITION BY category
        ORDER BY total_revenue DESC
    ) AS rank_in_category
FROM
(
    SELECT
        p.category,
        p.product_name,
        SUM(
            oi.quantity*oi.unit_price*
            (1-oi.discount_percent/100.0)
        ) AS total_revenue
    FROM products p
    JOIN order_items oi
    ON p.product_id=oi.product_id
    GROUP BY
        p.category,
        p.product_name
);

category,product_name,total_revenue,rank_in_category
Books,Comics,362238.6221000001,1
Books,Magazine,342311.17329999997,2
Books,Novel,293283.1039000001,3
Books,Dictionary,275198.14849999995,4
Books,Biography,261547.35039999997,5
Clothing,Shirt,283327.6293,1
Clothing,Dress,249955.70880000005,2
Clothing,Jacket,225625.92740000004,3
Clothing,Jeans,204954.21509999997,4
Clothing,T-Shirt,93306.85929999998,5


### LAG / LEAD Analysis
### 
### Show:
### 
### customer_id
### order_date
### previous_order_date
### days_gap
### 
### Flag customers whose average gap is more than 30 days as "At Risk".

In [0]:
%sql
WITH customer_orders AS
(
SELECT
    customer_id,
    order_date,
    LAG(order_date) OVER(
        PARTITION BY customer_id
        ORDER BY order_date
    ) AS previous_order_date
FROM orders
),

gaps AS
(
SELECT
    customer_id,
    order_date,
    previous_order_date,
    DATEDIFF(
        TO_DATE(order_date),
        TO_DATE(previous_order_date)
    ) AS days_gap
FROM customer_orders
)

SELECT
    customer_id,
    order_date,
    previous_order_date,
    days_gap,
    CASE
        WHEN AVG(days_gap) OVER(PARTITION BY customer_id) > 30
        THEN 'At Risk'
        ELSE 'Active'
    END AS customer_status
FROM gaps;

customer_id,order_date,previous_order_date,days_gap,customer_status
0.0,2024-07-19 06:57:52,null,null,Active
0.0,2024-07-24 08:03:03,2024-07-19 06:57:52,5,Active
0.0,2024-08-14 10:08:05,2024-07-24 08:03:03,21,Active
0.0,2024-09-13 09:24:07,2024-08-14 10:08:05,30,Active
0.0,2024-10-13 12:26:13,2024-09-13 09:24:07,30,Active
0.0,2024-11-10 00:46:03,2024-10-13 12:26:13,28,Active
0.0,2024-11-15 08:17:26,2024-11-10 00:46:03,5,Active
0.0,2024-11-27 14:15:36,2024-11-15 08:17:26,12,Active
0.0,2024-12-02 14:40:08,2024-11-27 14:15:36,5,Active
0.0,2025-01-01 18:56:08,2024-12-02 14:40:08,30,Active
